# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We will review its Croissant schema, inspect the available record sets and fields—**referencing everything by their `@id`**—and perform basic analysis and visualization.

### Dataset Source

The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset Croissant package
dataset = mlc.Dataset(croissant_url)
# Get the Croissant metadata object (do not treat as a dict)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

# Optionally, preview full metadata as JSON-LD
# print(json.dumps(metadata.to_json(), indent=2))

## 2. Data Overview
Review available **record sets** (`@id`), **fields/columns** (`@id`), and their structure from the Croissant metadata.

> **Note:** All entities are referenced by their `@id`. This step helps identify which record sets and fields you might want to extract.

In [ ]:
# Display all available record sets by @id and list their fields/columns
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs_obj in metadata.record_sets:
        print(f"\nRecord set: {rs_obj.name} (@id: {rs_obj.id})")
        if hasattr(rs_obj, 'fields'):
            for field in rs_obj.fields:
                print(f"  Field: {getattr(field, 'name', '')} (@id: {field.id}, type: {getattr(field, 'data_type', 'n/a')})")
else:
    print("No record sets found in metadata.")

In [ ]:
# Alternatively, enumerate the records of the main record set by @id (replace below with actual record set @id from overview if needed)
# To list all available record set @id values:
record_sets = getattr(metadata, 'record_sets', None)

if record_sets and len(record_sets) > 0:
    first_record_set_id = record_sets[0].id
    print(f"\nEnumerating records from record set: {first_record_set_id}\n---")
    for i, x in enumerate(dataset.records(record_set=first_record_set_id)):
        if i >= 3:
            print(f"... (showing first 3 records)")
            break
        print(json.dumps(x, indent=2))
else:
    print("No record sets present.")

## 3. Data Extraction
Extract all data from each record set into a pandas DataFrame for downstream analysis.

You can select specific record sets and field `@id`s as desired. For this notebook, we extract all record sets by their `@id`.

In [ ]:
record_sets = getattr(metadata, 'record_sets', [])

dataframes = dict()

for record_set in record_sets:
    record_set_id = record_set.id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set: {record_set_id}")
    print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")

# As an example, preview the first DataFrame
if len(dataframes) > 0:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows of DataFrame for record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common processing steps: select a numeric field (by its `@id`), filter, normalize, and group data.

> *Update `numeric_field_id` and `group_field_id` based on your record set overview.*

In [ ]:
# Example: pick the first record set and select a numeric field from its columns for EDA

# Auto-select record set and field ids for demonstration:
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to infer a numeric field by dtype
    numeric_cols = df.select_dtypes(include=[int, float]).columns.tolist()
    if len(numeric_cols) == 0:
        print(f"No numeric columns found in record set {record_set_id}.")
        numeric_field_id = None
    else:
        numeric_field_id = numeric_cols[0] # Use the first numeric field for demo

    # Pick groupable fields: categorical or object fields
    group_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(group_candidates) == 0:
        print(f"No groupable/categorical columns found in record set {record_set_id}.")
        group_field_id = None
    else:
        group_field_id = group_candidates[0]

    # Filter records
    if numeric_field_id:
        threshold = df[numeric_field_id].quantile(0.8) if df[numeric_field_id].nunique() > 10 else df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No suitable record set loaded for EDA.")

## 5. Visualization
Visualize the distribution and relationships for a record set field. Update field IDs as meaningful for your data.

In [ ]:
# Plot numeric field's distribution and box by group (if available) for the same record set.
if len(dataframes) > 0 and numeric_field_id:
    # Histogram for numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Boxplot by group, if possible
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        order = df[group_field_id].value_counts().index[:10]
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id, order=order)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=35)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and perform initial processing and visual exploration of a complex clinical dataset using `mlcroissant`—referencing all dataset elements by their Croissant `@id`. The Croissant metadata allows for reproducible, schema-driven data exploration and integration.

You can extend this template for more advanced analyses or to work with additional record sets and fields as needed.